In [1]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
测试ToolManager：初始化全部离线工具和Point工具，获取instructions和prompt
"""

import os
import sys
from tool_server.tool_workers.tool_manager.base_manager import ToolManager
from tool_server.utils.utils import load_json_file

def main():
    """测试ToolManager的初始化和指令获取功能"""
    print("=== 测试ToolManager初始化与指令获取 ===\n")
    
    # 1. 获取所有可用的离线工具
    print("初始化所有离线工具和Point工具...")
    
    # 创建ToolManager实例，指定需要初始化的工具
    # 包括所有离线工具和Point在线工具
    manager = ToolManager(tools=["AStarWithPixelCoordinate", "Draw2DPath", 
                                "TurnCoordinateIntoTextMap", "Point"])
    
    # 2. 打印初始化结果
    print("\n初始化结果:")
    print(f"- 离线工具: {manager.available_offline_tools}")
    print(f"- 在线工具: {manager.available_online_tools}")
    print(f"- 所有可用工具: {manager.available_tools}")
    
    # 3. 获取所有工具的instructions
    print("\n获取工具指令...")
    tool_instructions = manager.get_tool_instructions()
    
    # 4. 打印每个工具的instruction
    print("\n=== 工具指令详情 ===")
    for tool_name, instruction in tool_instructions.items():
        print(f"\n--- {tool_name} ---")
        # 如果指令太长，只打印前200个字符
        if len(instruction) > 200:
            print(f"{instruction[:200]}...")
        else:
            print(instruction)
    
    # 5. 获取并打印工具prompt
    print("\n=== 获取工具Prompt ===")
    prompt = manager.get_tool_prompt(prompt_type="one_tool_call")
    
    # 打印prompt的前300个和后300个字符
    print("\nPrompt头部:")
    print(prompt[:300] + "...")
    
    print("\nPrompt尾部:")
    print("..." + prompt[-300:])
    
    # 6. 保存完整prompt到文件
    prompt_file = "tool_prompt.txt"
    with open(prompt_file, 'w', encoding='utf-8') as f:
        f.write(prompt)
    print(f"\n完整Prompt已保存到文件: {os.path.abspath(prompt_file)}")
    
    # 7. 测试获取特定工具的prompt
    print("\n=== 获取特定工具的Prompt ===")
    astar_prompt = manager.get_tool_prompt(
        prompt_type="one_tool_call", 
        tools=["AStarWithPixelCoordinate"]
    )
    
    print(f"\nA*寻路工具Prompt (前200字符):")
    print(astar_prompt[:200] + "...")
    
    # 8. 测试总结
    print("\n=== 测试总结 ===")
    print(f"成功初始化工具总数: {len(manager.available_tools)}")
    print(f"获取到的工具指令数量: {len(tool_instructions)}")
    
    if "Point" in manager.available_online_tools:
        print("✅ 成功初始化Point工具")
    else:
        print("❌ Point工具初始化失败")
        
    if set(manager.available_offline_tools) == set(["AStarWithPixelCoordinate", "Draw2DPath", "TurnCoordinateIntoTextMap"]):
        print("✅ 成功初始化所有离线工具")
    else:
        print("❌ 离线工具初始化不完整")
    
    print("\n测试完成!")



2025-07-20 01:01:49 | INFO | base_offline_worker | Initialized offline worker: AStarWithPixelCoordinate
2025-07-20 01:01:49 | INFO | base_offline_worker | Initialized offline worker: Draw2DPath
2025-07-20 01:01:49 | INFO | base_offline_worker | Initialized offline worker: TurnCoordinateIntoTextMap


In [1]:
import os
import sys
from tool_server.tool_workers.tool_manager.base_manager import ToolManager
from tool_server.utils.utils import load_json_file
manager = ToolManager(tools=["AStarWithPixelCoordinate", "Draw2DPath", 
                                "TurnCoordinateIntoTextMap", "Point"])

2025-07-20 01:19:54 | INFO | base_offline_worker | Initialized offline worker: AStarWithPixelCoordinate
2025-07-20 01:19:54 | INFO | base_offline_worker | Initialized offline worker: Draw2DPath
2025-07-20 01:19:54 | INFO | base_offline_worker | Initialized offline worker: TurnCoordinateIntoTextMap
2025-07-20 01:19:54 | INFO | tool_manager | Offline Tools: ['AStarWithPixelCoordinate', 'Draw2DPath', 'TurnCoordinateIntoTextMap']
2025-07-20 01:19:54 | INFO | tool_manager | controller_addr is None, using default from controller_addr_location
2025-07-20 01:19:54 | INFO | tool_manager | Online Tools: ['Point']
2025-07-20 01:19:54 | INFO | tool_manager | Not all required online tools are prepared successfully, missing: ['AStarWithPixelCoordinate', 'Draw2DPath', 'TurnCoordinateIntoTextMap']
2025-07-20 01:19:54 | INFO | tool_manager | ToolManager is initialized.
2025-07-20 01:19:54 | INFO | stdout | available_tools ['Point', 'AStarWithPixelCoordinate', 'Draw2DPath', 'TurnCoordinateIntoTextMap']


In [2]:
manager.available_tools

['Point',
 'AStarWithPixelCoordinate',
 'Draw2DPath',
 'TurnCoordinateIntoTextMap']

In [4]:
manager.get_tool_instructions()

{'AStarWithPixelCoordinate': {'type': 'function',
  'function': {'name': 'AStarWithPixelCoordinate',
   'description': 'Find the shortest path from start to goal while avoiding obstacles using A* algorithm',
   'parameters': {'type': 'object',
    'properties': {'start': {'type': 'array',
      'description': 'Starting point coordinates [x, y] in pixels, e.g., [100, 200]'},
     'goal': {'type': 'array',
      'description': 'Goal point coordinates [x, y] in pixels, e.g., [300, 400]'},
     'obstacles': {'type': 'array',
      'description': 'Array of obstacle coordinates [[x1, y1], [x2, y2], ...] in pixels, e.g., [[150, 150], [200, 250], [300, 300]]'}},
    'required': ['start', 'goal', 'obstacles']},
   'returns': {'type': 'object',
    'properties': {'status': {'type': 'string',
      'description': "Status of the pathfinding operation ('success' or 'failed')"},
     'path': {'type': 'string',
      'description': "Control sequence as comma-separated uppercase directions (L,R,U,D), 

In [3]:
a = manager.get_tool_prompt(prompt_type="one_tool_call")
# Write the tool prompt to a file
output_path = "./tool_prompt.txt"
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(a)
print(f"Tool prompt has been written to: {output_path}")

2025-07-20 01:19:58 | INFO | stdout | Tool prompt has been written to: ./tool_prompt.txt


In [12]:
print(f"a")